# Notebook 00 — ดาวน์โหลดและจัดเตรียมข้อมูล Sounding สถานีเชียงใหม่ (WMO 48327)

**รายวิชา:** บรรยากาศและคุณภาพอากาศ / Atmospheric Environment and Air Quality  
**ระดับ:** นิสิตสิ่งแวดล้อม ปี 3–4  
**สถานี:** Chiang Mai — WMO ID **48327**  
**ช่วงศึกษา:** **1 มีนาคม–30 เมษายน 2024**  
**แหล่งข้อมูล:** University of Wyoming Atmospheric Science Radiosonde Archive  
**แพลตฟอร์ม:** Google Colab + Google Drive

## วัตถุประสงค์

Notebook นี้เป็น **data-preparation notebook** ก่อนเข้าสู่ Notebook 01–04 เพื่อให้นิสิตเข้าใจการได้มาของข้อมูล การตรวจสอบ data availability การเก็บ raw/processed data และ metadata อย่างเป็นระบบ

ผลลัพธ์หลักของ Notebook 00 คือ

1. sounding ราย launch ในรูป CSV
2. raw response จาก Wyoming
3. `launch_manifest.csv` สำหรับตรวจวัน/เวลาที่มีและไม่มีข้อมูล
4. combined long-format dataset แบบ `.csv.gz`
5. station metadata
6. ZIP archive สำหรับ provenance
7. โฟลเดอร์ `github_ready` สำหรับนำข้อมูลชุดกลางขึ้น GitHub

> Notebook นี้ยังไม่เน้น CAPE, CIN, inversion หรือ PBL height การวิเคราะห์เชิงกายภาพจะเริ่มใน Notebook 01–04 หลังจากตรวจ coverage ของข้อมูลจริงแล้ว

## 0.1 เวลา UTC และเวลาไทย

ประเทศไทยใช้ **ICT = UTC+7**

- **00 UTC = 07:00 ICT** — ช่วงเช้า
- **06 UTC = 13:00 ICT** — ช่วงบ่าย
- **12 UTC = 19:00 ICT** — ช่วงหัวค่ำ

หน้า archive ของ Wyoming สามารถ query หลาย synoptic hour ได้ แต่ไม่ได้หมายความว่าสถานี 48327 จะมี observation ทุกเวลา ดังนั้น notebook นี้แยกสถานะอย่างชัดเจนเป็น

- `downloaded` — ดาวน์โหลดและ parse สำเร็จ
- `cached` — มีไฟล์อยู่แล้ว จึงไม่ดาวน์โหลดซ้ำ
- `no_data` — server ตอบกลับ แต่ไม่มี sounding เวลานั้น
- `error` — network หรือ parser error

**หลักการสำคัญ:** `no_data` ไม่เท่ากับ `code error`

In [ ]:
# ================================================================
# CELL 1 — Install packages for Google Colab
# ================================================================
!pip -q install pandas numpy requests tqdm

In [ ]:
# ================================================================
# CELL 2 — Mount Google Drive
# ================================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


## 0.2 กำหนดช่วงเวลาและชั่วโมงที่ต้องการตรวจ

ค่าเริ่มต้นจะตรวจ 00, 06 และ 12 UTC เพื่อให้ทราบ availability จริงในช่วงมีนาคม–เมษายน 2024

หากต้องการเฉพาะข้อมูลหลัก 00 UTC ให้ตั้ง

```python
CHECK_OPTIONAL_HOURS = False
```

โค้ดยังคงใช้ Wyoming WSGI endpoint ตาม notebook ต้นแบบของอาจารย์ (`src=FM35`, `type=TEXT:LIST`) แต่ปรับ parser ให้รองรับ fixed-width table และ missing values ได้ดีขึ้น

In [ ]:
# ================================================================
# CELL 3 — Configuration
# ================================================================
from pathlib import Path
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import pandas as pd

STATION_ID = "48327"
STATION_NAME_EXPECTED = "Chiang Mai"

START_DATE = "2024-03-01"
END_DATE   = "2024-04-30"

PRIMARY_HOURS_UTC = [0]
OPTIONAL_HOURS_UTC = [6, 12]   # 06Z = 13:00 ICT; 12Z = 19:00 ICT
CHECK_OPTIONAL_HOURS = True

HOURS_UTC = (
    PRIMARY_HOURS_UTC + OPTIONAL_HOURS_UTC
    if CHECK_OPTIONAL_HOURS
    else PRIMARY_HOURS_UTC
)

MAX_RETRIES = 4
RETRY_DELAY_SECONDS = 4
REQUEST_DELAY_SECONDS = 1.0
REQUEST_TIMEOUT_SECONDS = 60

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Teaching_AirQuality/"
    "ChiangMai_Sounding_48327_MarApr2024"
)

UTC = timezone.utc
ICT = ZoneInfo("Asia/Bangkok")

dates = pd.date_range(START_DATE, END_DATE, freq="D")
n_requests = len(dates) * len(HOURS_UTC)

print("Station:", STATION_ID, "-", STATION_NAME_EXPECTED)
print("Period :", START_DATE, "to", END_DATE)
print("UTC hours checked:", HOURS_UTC)
print("Potential launch requests:", n_requests)
print("Project directory:", PROJECT_DIR)

Station: 48327 - Chiang Mai
Period : 2024-03-01 to 2024-04-30
UTC hours checked: [0, 6, 12]
Potential launch requests: 183
Project directory: /content/drive/MyDrive/Teaching_AirQuality/ChiangMai_Sounding_48327_MarApr2024


## 0.3 โครงสร้างโฟลเดอร์

```text
ChiangMai_Sounding_48327_MarApr2024/
├── 00_raw_text/       # raw response จาก Wyoming
├── 01_profiles_csv/   # sounding ราย launch
├── 02_processed/      # combined .csv.gz
├── 03_metadata/       # manifest, station metadata, report
├── 04_archive/        # ZIP raw + profiles
└── 05_github_ready/   # ไฟล์ที่แนะนำให้นำขึ้น GitHub
```

โครงสร้างนี้ช่วยสอนแนวคิด **raw → processed → metadata → distribution** และทำให้ reproducibility ชัดเจน

In [ ]:
# ================================================================
# CELL 4 — Create project folders
# ================================================================
RAW_DIR       = PROJECT_DIR / "00_raw_text"
PROFILE_DIR   = PROJECT_DIR / "01_profiles_csv"
PROCESSED_DIR = PROJECT_DIR / "02_processed"
META_DIR      = PROJECT_DIR / "03_metadata"
ARCHIVE_DIR   = PROJECT_DIR / "04_archive"
GITHUB_DIR    = PROJECT_DIR / "05_github_ready"

for folder in [RAW_DIR, PROFILE_DIR, PROCESSED_DIR, META_DIR, ARCHIVE_DIR, GITHUB_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Created/checked folders:")
for folder in [RAW_DIR, PROFILE_DIR, PROCESSED_DIR, META_DIR, ARCHIVE_DIR, GITHUB_DIR]:
    print(" -", folder)

Created/checked folders:
 - /content/drive/MyDrive/Teaching_AirQuality/ChiangMai_Sounding_48327_MarApr2024/00_raw_text
 - /content/drive/MyDrive/Teaching_AirQuality/ChiangMai_Sounding_48327_MarApr2024/01_profiles_csv
 - /content/drive/MyDrive/Teaching_AirQuality/ChiangMai_Sounding_48327_MarApr2024/02_processed
 - /content/drive/MyDrive/Teaching_AirQuality/ChiangMai_Sounding_48327_MarApr2024/03_metadata
 - /content/drive/MyDrive/Teaching_AirQuality/ChiangMai_Sounding_48327_MarApr2024/04_archive
 - /content/drive/MyDrive/Teaching_AirQuality/ChiangMai_Sounding_48327_MarApr2024/05_github_ready


## 0.4 ตัวแปรใน Wyoming sounding

ตารางหลักมีตัวแปรดังนี้

| Wyoming | ความหมาย | หน่วย |
|---|---|---|
| PRES | Pressure | hPa |
| HGHT | Geopotential height | m |
| TEMP | Temperature | °C |
| DWPT | Dew-point temperature | °C |
| RELH | Relative humidity | % |
| MIXR | Mixing ratio | g kg⁻¹ |
| DRCT | Wind direction | degree |
| SPED | Wind speed ใน WSGI รุ่นใหม่ | m s⁻¹ |
| THTA | Potential temperature | K |
| THTE | Equivalent potential temperature | K |
| THTV | Virtual potential temperature | K |

ข้อมูลเป็น **fixed-width table** และบางระดับอาจมีค่าว่างในคอลัมน์ภายใน ดังนั้น notebook นี้ไม่ใช้เพียง `line.split()` เพราะ missing value อาจทำให้ตำแหน่งคอลัมน์เลื่อน

Parser จะรักษาค่าว่างและสร้าง `wind_speed_ms` เป็นหน่วยมาตรฐานสำหรับ Notebook ต่อไป ขณะเดียวกันยังเก็บค่าต้นฉบับใน `wind_speed_source`

In [ ]:
# ================================================================
# CELL 5 — Downloader, parser and structural QC functions
# ================================================================
import re
import html
import time
import shutil
import zipfile
from typing import Optional, Tuple, Dict, Any

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

WYOMING_ENDPOINT = "https://weather.uwyo.edu/wsgi/sounding"

STANDARD_COLUMNS = [
    "pressure_hPa", "height_m", "temperature_C", "dewpoint_C",
    "relative_humidity_pct", "mixing_ratio_gkg", "wind_direction_deg",
    "wind_speed_source", "theta_K", "theta_e_K", "theta_v_K",
]

NO_DATA_PATTERNS = [
    "can't get", "cannot get", "no data", "no sounding",
    "not available", "sorry",
]


def to_plain_text(raw_text: str) -> str:
    """Remove HTML tags while preserving line structure."""
    text = html.unescape(raw_text)
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.I)
    text = re.sub(r"</p\s*>", "\n", text, flags=re.I)
    text = re.sub(r"</h[1-6]\s*>", "\n", text, flags=re.I)
    text = re.sub(r"<[^>]+>", "", text)
    return text.replace("\r\n", "\n").replace("\r", "\n")


def _float_or_nan(value: str) -> float:
    value = str(value).strip()
    if value == "":
        return np.nan
    try:
        return float(value)
    except Exception:
        return np.nan


def detect_wind_unit(header_fields, unit_fields) -> str:
    header = str(header_fields[7]).strip().upper() if len(header_fields) >= 8 else ""
    unit = str(unit_fields[7]).strip().lower() if len(unit_fields) >= 8 else ""

    if "m/s" in unit or header in {"SPED", "SPEED"}:
        return "m/s"
    if "knot" in unit or unit in {"kt", "kts"} or header == "SKNT":
        return "knot"
    return "unknown"


def parse_wyoming_table(raw_text: str) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Parse Wyoming TEXT:LIST fixed-width sounding table."""
    plain = to_plain_text(raw_text)
    lines = plain.splitlines()

    header_idx = None
    for i, line in enumerate(lines):
        if (
            re.search(r"\bPRES\b", line)
            and re.search(r"\bHGHT\b", line)
            and re.search(r"\bTEMP\b", line)
            and (re.search(r"\bSPED\b", line) or re.search(r"\bSKNT\b", line))
        ):
            header_idx = i
            break

    if header_idx is None:
        raise ValueError("Could not locate Wyoming sounding table header.")

    header_line = lines[header_idx]
    units_line = lines[header_idx + 1] if header_idx + 1 < len(lines) else ""

    # Wyoming table is seven characters per field in the standard listing.
    width = 7
    n_fields = 11
    header_fields = [header_line[i*width:(i+1)*width].strip() for i in range(n_fields)]
    unit_fields = [units_line[i*width:(i+1)*width].strip() for i in range(n_fields)]
    wind_unit = detect_wind_unit(header_fields, unit_fields)

    data_start = header_idx + 2
    while data_start < len(lines) and lines[data_start].strip() and set(lines[data_start].strip()) <= {"-"}:
        data_start += 1

    rows = []
    for line in lines[data_start:]:
        first_field = line[0:width].strip()
        if first_field == "":
            if rows:
                continue
            continue

        try:
            float(first_field)
        except Exception:
            if rows:
                break
            continue

        values = [_float_or_nan(line[i*width:(i+1)*width]) for i in range(n_fields)]
        rows.append(values)

    if len(rows) < 5:
        raise ValueError(f"Parsed only {len(rows)} sounding levels.")

    df = pd.DataFrame(rows, columns=STANDARD_COLUMNS)
    df = df.dropna(subset=["pressure_hPa"]).reset_index(drop=True)

    if wind_unit == "m/s":
        df["wind_speed_ms"] = df["wind_speed_source"]
    elif wind_unit == "knot":
        df["wind_speed_ms"] = df["wind_speed_source"] * 0.514444
    else:
        df["wind_speed_ms"] = np.nan

    df["wind_speed_source_unit"] = wind_unit

    return df, {
        "detected_header_fields": " | ".join(header_fields),
        "detected_unit_fields": " | ".join(unit_fields),
        "wind_speed_source_unit": wind_unit,
    }


def parse_station_metadata(raw_text: str) -> Dict[str, Any]:
    """Extract station name, latitude, longitude and elevation when available."""
    plain = to_plain_text(raw_text)
    meta = {
        "station_name_source": None,
        "latitude": np.nan,
        "longitude": np.nan,
        "elevation_m": np.nan,
    }

    m_name = re.search(r"<h3[^>]*>(.*?)</h3>", raw_text, flags=re.I | re.S)
    if m_name:
        name = re.sub(r"<[^>]+>", "", m_name.group(1))
        meta["station_name_source"] = html.unescape(name).strip()

    m_latlon = re.search(
        r"Latitude:\s*([-+]?\d+(?:\.\d+)?)\s+Longitude:\s*([-+]?\d+(?:\.\d+)?)",
        plain, flags=re.I,
    )
    if m_latlon:
        meta["latitude"] = float(m_latlon.group(1))
        meta["longitude"] = float(m_latlon.group(2))

    m_elev = re.search(r"Station\s+elevation:\s*([-+]?\d+(?:\.\d+)?)", plain, flags=re.I)
    if m_elev:
        meta["elevation_m"] = float(m_elev.group(1))

    return meta


def profile_qc_summary(df: pd.DataFrame) -> Dict[str, Any]:
    """Basic structural QC only; scientific QC is deferred to later notebooks."""
    p = pd.to_numeric(df["pressure_hPa"], errors="coerce")
    z = pd.to_numeric(df["height_m"], errors="coerce")
    p_valid = p.dropna()

    pressure_monotonic = bool((p_valid.diff().dropna() <= 0).all()) if len(p_valid) >= 2 else False

    return {
        "n_levels": int(len(df)),
        "surface_pressure_hPa": float(p_valid.iloc[0]) if len(p_valid) else np.nan,
        "top_pressure_hPa": float(p_valid.iloc[-1]) if len(p_valid) else np.nan,
        "max_height_m": float(z.max()) if z.notna().any() else np.nan,
        "pressure_monotonic_decreasing": pressure_monotonic,
        "temperature_valid_n": int(df["temperature_C"].notna().sum()),
        "dewpoint_valid_n": int(df["dewpoint_C"].notna().sum()),
        "wind_valid_n": int(df["wind_speed_source"].notna().sum()),
    }


def launch_tag(dt_utc: datetime) -> str:
    return dt_utc.strftime("%Y-%m-%d_%HZ")


def launch_paths(dt_utc: datetime):
    tag = launch_tag(dt_utc)
    return (
        RAW_DIR / f"sounding_{STATION_ID}_{tag}.txt",
        PROFILE_DIR / f"sounding_{STATION_ID}_{tag}.csv",
    )


session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (educational radiosonde download; Google Colab teaching workflow)"
})


def fetch_wyoming_wsgi(dt_utc: datetime, station_id: str, max_retries: int = MAX_RETRIES):
    """Fetch one sounding. Return (raw_text, error_message)."""
    params = {
        "datetime": dt_utc.strftime("%Y-%m-%d %H:00:00"),
        "id": station_id,
        "src": "FM35",
        "type": "TEXT:LIST",
    }

    last_error = None
    for attempt in range(max_retries):
        try:
            r = session.get(WYOMING_ENDPOINT, params=params, timeout=REQUEST_TIMEOUT_SECONDS)

            if 400 <= r.status_code < 500:
                body = r.text or ""
                if any(pattern in body.lower() for pattern in NO_DATA_PATTERNS):
                    return None, f"no data (HTTP {r.status_code})"
                return None, f"HTTP {r.status_code}"

            r.raise_for_status()
            raw_text = r.text
            lower = raw_text.lower()

            if any(pattern in lower for pattern in NO_DATA_PATTERNS):
                has_table = "PRES" in raw_text and "HGHT" in raw_text and ("SPED" in raw_text or "SKNT" in raw_text)
                if not has_table:
                    return None, "no data at this time"

            return raw_text, None

        except Exception as exc:
            last_error = str(exc)[:220]
            if attempt < max_retries - 1:
                time.sleep(RETRY_DELAY_SECONDS * (2 ** attempt))

    return None, last_error or "unknown request error"


def download_one_launch(dt_utc: datetime, force: bool = False, verbose: bool = False):
    """Download, parse, QC and cache one launch."""
    raw_file, csv_file = launch_paths(dt_utc)
    local_dt = dt_utc.astimezone(ICT)

    base = {
        "station_id": STATION_ID,
        "requested_datetime_utc": dt_utc.strftime("%Y-%m-%d %H:%M:%S"),
        "datetime_ict": local_dt.strftime("%Y-%m-%d %H:%M:%S"),
        "date_utc": dt_utc.strftime("%Y-%m-%d"),
        "hour_utc": dt_utc.hour,
        "hour_ict": local_dt.hour,
        "status": None,
        "message": None,
        "n_levels": np.nan,
        "surface_pressure_hPa": np.nan,
        "top_pressure_hPa": np.nan,
        "max_height_m": np.nan,
        "pressure_monotonic_decreasing": np.nan,
        "temperature_valid_n": np.nan,
        "dewpoint_valid_n": np.nan,
        "wind_valid_n": np.nan,
        "wind_speed_source_unit": None,
        "raw_text_file": str(raw_file),
        "profile_csv_file": str(csv_file),
        "source_endpoint": WYOMING_ENDPOINT,
    }

    if csv_file.exists() and not force:
        try:
            df = pd.read_csv(csv_file)
            rec = base.copy()
            rec.update(profile_qc_summary(df))
            rec["status"] = "cached"
            rec["message"] = "existing profile CSV reused"

            if "wind_speed_source_unit" in df.columns:
                u = df["wind_speed_source_unit"].dropna().astype(str).unique().tolist()
                rec["wind_speed_source_unit"] = u[0] if u else None

            meta = {}
            if raw_file.exists():
                meta = parse_station_metadata(raw_file.read_text(encoding="utf-8", errors="ignore"))

            if verbose:
                print(f"CACHED: {launch_tag(dt_utc)} ({len(df)} levels)")
            return rec, meta
        except Exception:
            pass

    raw_text, err = fetch_wyoming_wsgi(dt_utc, STATION_ID)

    if raw_text is None:
        rec = base.copy()
        rec["status"] = "no_data" if "no data" in (err or "").lower() else "error"
        rec["message"] = err
        if verbose:
            print(f"{rec['status'].upper()}: {launch_tag(dt_utc)} - {err}")
        return rec, {}

    raw_file.write_text(raw_text, encoding="utf-8")

    try:
        df, parser_meta = parse_wyoming_table(raw_text)
        station_meta = parse_station_metadata(raw_text)
        qc = profile_qc_summary(df)

        df.insert(0, "station_id", STATION_ID)
        df.insert(1, "launch_datetime_utc", dt_utc.strftime("%Y-%m-%d %H:%M:%S"))
        df.insert(2, "launch_datetime_ict", local_dt.strftime("%Y-%m-%d %H:%M:%S"))
        df.insert(3, "hour_utc", dt_utc.hour)
        df.insert(4, "hour_ict", local_dt.hour)
        df.to_csv(csv_file, index=False)

        rec = base.copy()
        rec.update(qc)
        rec["status"] = "downloaded"
        rec["message"] = "downloaded and parsed successfully"
        rec["wind_speed_source_unit"] = parser_meta["wind_speed_source_unit"]

        if verbose:
            print(f"DOWNLOADED: {launch_tag(dt_utc)} | {len(df)} levels | wind unit={parser_meta['wind_speed_source_unit']}")
        return rec, station_meta

    except Exception as exc:
        rec = base.copy()
        rec["status"] = "error"
        rec["message"] = f"parse error: {str(exc)[:220]}"
        if verbose:
            print(f"PARSE ERROR: {launch_tag(dt_utc)} - {exc}")
        return rec, parse_station_metadata(raw_text)

## 0.5 Test launch ก่อน batch download

ทดสอบ **1 มีนาคม 2024 เวลา 00 UTC** ก่อน เพื่อดูว่า endpoint และ parser ทำงานได้หรือไม่

สิ่งที่ควรตรวจ:

- `status` เป็น `downloaded` หรือ `cached`
- `n_levels` มีหลายระดับ
- pressure ลดลงจากใกล้พื้นสู่ระดับบน
- `wind_speed_source_unit` ถูกตรวจเป็น `m/s`
- ตาราง 10 ระดับแรกดูสมเหตุสมผล

ถ้า test launch เป็น `error` ควรตรวจข้อความ error ก่อนรัน batch ทั้งช่วง

In [ ]:
# ================================================================
# CELL 6 — Test one launch: 1 March 2024, 00 UTC
# ================================================================
TEST_DT = datetime(2024, 3, 1, 0, tzinfo=UTC)

test_record, test_meta = download_one_launch(TEST_DT, force=False, verbose=True)
display(pd.DataFrame([test_record]))

test_csv = Path(test_record["profile_csv_file"])

if test_record["status"] in {"downloaded", "cached"} and test_csv.exists():
    test_df = pd.read_csv(test_csv)
    print("\nStation metadata parsed from source:")
    print(test_meta)
    print("\nFirst 10 vertical levels:")
    display(test_df.head(10))
    print("\nColumns:")
    print(test_df.columns.tolist())
else:
    print("\nTest launch was not successfully parsed.")
    print("Check the status/message above before running the full batch.")

DOWNLOADED: 2024-03-01_00Z | 46 levels | wind unit=m/s


,station_id,requested_datetime_utc,datetime_ict,date_utc,hour_utc,hour_ict,status,message,n_levels,surface_pressure_hPa,top_pressure_hPa,max_height_m,pressure_monotonic_decreasing,temperature_valid_n,dewpoint_valid_n,wind_valid_n,wind_speed_source_unit,raw_text_file,profile_csv_file,source_endpoint
0,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,2024-03-01,0,7,downloaded,downloaded and parsed successfully,46,1000.0,749.0,2812.0,True,46,46,1,m/s,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding



Station metadata parsed from source:
{'station_name_source': 'CHIANG MAI, THAILAND', 'latitude': 18.78, 'longitude': 98.98, 'elevation_m': nan}

First 10 vertical levels:


,station_id,launch_datetime_utc,launch_datetime_ict,hour_utc,hour_ict,pressure_hPa,height_m,temperature_C,dewpoint_C,relative_humidity_pct,mixing_ratio_gkg,wind_direction_deg,wind_speed_source,theta_K,theta_e_K,theta_v_K,wind_speed_ms,wind_speed_source_unit
0,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,0,7,1000.0,314.0,20.8,19.8,94.0,14.68,0.0,0.0,293.9,336.0,296.5,0.0,m/s
1,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,0,7,968.0,597.0,22.4,14.4,61.0,10.71,NaN,NaN,298.3,329.7,300.2,NaN,m/s
2,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,0,7,962.0,651.0,23.0,15.0,61.0,11.21,NaN,NaN,299.4,332.5,301.5,NaN,m/s
3,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,0,7,957.0,696.0,23.6,14.6,57.0,10.98,NaN,NaN,300.5,333.0,302.5,NaN,m/s
4,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,0,7,951.0,751.0,24.0,14.0,54.0,10.62,NaN,NaN,301.4,333.0,303.4,NaN,m/s
5,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,0,7,946.0,798.0,24.4,13.4,50.0,10.26,NaN,NaN,302.3,333.0,304.2,NaN,m/s
6,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,0,7,940.0,853.0,24.6,12.6,47.0,9.79,NaN,NaN,303.1,332.5,304.8,NaN,m/s
7,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,0,7,935.0,900.0,25.0,11.0,41.0,8.85,NaN,NaN,303.9,330.7,305.6,NaN,m/s
8,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,0,7,930.0,947.0,25.0,10.0,39.0,8.31,NaN,NaN,304.4,329.6,305.9,NaN,m/s
9,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,0,7,925.0,994.0,25.2,10.2,39.0,8.47,NaN,NaN,305.1,330.8,306.6,NaN,m/s



Columns:
['station_id', 'launch_datetime_utc', 'launch_datetime_ict', 'hour_utc', 'hour_ict', 'pressure_hPa', 'height_m', 'temperature_C', 'dewpoint_C', 'relative_humidity_pct', 'mixing_ratio_gkg', 'wind_direction_deg', 'wind_speed_source', 'theta_K', 'theta_e_K', 'theta_v_K', 'wind_speed_ms', 'wind_speed_source_unit']


## 0.6 Batch download: 1 มีนาคม–30 เมษายน 2024

Batch downloader จะ

1. วนตามวันที่และชั่วโมง UTC
2. reuse ไฟล์เดิมถ้ามี
3. retry เมื่อเป็น transient network error
4. แยก `no_data` จาก `error`
5. เว้นช่วงระหว่าง request เพื่อลดภาระ server
6. สร้าง manifest ครบทุก launch ที่ตรวจ

เมื่อ `CHECK_OPTIONAL_HOURS=True` เราจะได้ coverage ของ 00, 06 และ 12 UTC จากข้อมูลจริง

In [ ]:
# ================================================================
# CELL 7 — Full batch download
# ================================================================
launch_datetimes = []
for date in pd.date_range(START_DATE, END_DATE, freq="D"):
    for hour in HOURS_UTC:
        launch_datetimes.append(
            datetime(date.year, date.month, date.day, int(hour), tzinfo=UTC)
        )

records = []
station_meta_records = []

progress = tqdm(launch_datetimes, desc="Downloading/checking soundings", unit="launch")

for dt_utc in progress:
    record, station_meta = download_one_launch(dt_utc, force=False, verbose=False)
    records.append(record)

    if station_meta:
        station_meta_records.append({
            "requested_datetime_utc": dt_utc.strftime("%Y-%m-%d %H:%M:%S"),
            **station_meta,
        })

    progress.set_postfix({"UTC": f"{dt_utc.hour:02d}Z", "status": record["status"]})

    if record["status"] in {"downloaded", "no_data", "error"}:
        time.sleep(REQUEST_DELAY_SECONDS)

manifest = pd.DataFrame(records)
MANIFEST_FILE = META_DIR / f"launch_manifest_{STATION_ID}_20240301_20240430.csv"
manifest.to_csv(MANIFEST_FILE, index=False)

print("\nSaved manifest:")
print(MANIFEST_FILE)
display(manifest.head(12))

Downloading/checking soundings:   0%|          | 0/183 [00:00<?, ?launch/s]


Saved manifest:
/content/drive/MyDrive/Teaching_AirQuality/ChiangMai_Sounding_48327_MarApr2024/03_metadata/launch_manifest_48327_20240301_20240430.csv


,station_id,requested_datetime_utc,datetime_ict,date_utc,hour_utc,hour_ict,status,message,n_levels,surface_pressure_hPa,top_pressure_hPa,max_height_m,pressure_monotonic_decreasing,temperature_valid_n,dewpoint_valid_n,wind_valid_n,wind_speed_source_unit,raw_text_file,profile_csv_file,source_endpoint
0,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,2024-03-01,0,7,cached,existing profile CSV reused,46.0,1000.0,749.0,2812.0,True,46.0,46.0,1.0,m/s,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding
1,48327,2024-03-01 06:00:00,2024-03-01 13:00:00,2024-03-01,6,13,error,HTTP 404,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding
2,48327,2024-03-01 12:00:00,2024-03-01 19:00:00,2024-03-01,12,19,error,HTTP 404,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding
3,48327,2024-03-02 00:00:00,2024-03-02 07:00:00,2024-03-02,0,7,downloaded,downloaded and parsed successfully,53.0,1000.0,700.0,2850.0,True,53.0,53.0,53.0,m/s,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding
4,48327,2024-03-02 06:00:00,2024-03-02 13:00:00,2024-03-02,6,13,error,HTTP 404,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding
5,48327,2024-03-02 12:00:00,2024-03-02 19:00:00,2024-03-02,12,19,error,HTTP 404,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding
6,48327,2024-03-03 00:00:00,2024-03-03 07:00:00,2024-03-03,0,7,downloaded,downloaded and parsed successfully,63.0,1000.0,700.0,2850.0,True,63.0,63.0,63.0,m/s,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding
7,48327,2024-03-03 06:00:00,2024-03-03 13:00:00,2024-03-03,6,13,error,HTTP 404,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding
8,48327,2024-03-03 12:00:00,2024-03-03 19:00:00,2024-03-03,12,19,error,HTTP 404,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding
9,48327,2024-03-04 00:00:00,2024-03-04 07:00:00,2024-03-04,0,7,downloaded,downloaded and parsed successfully,56.0,1000.0,700.0,3450.0,True,56.0,56.0,56.0,m/s,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding


## 0.7 Data availability

ใช้ตัวชี้วัดพื้นฐาน

\[
Availability(\%) = \frac{N_{success}}{N_{requested}}\times 100
\]

โดย `success` รวม `downloaded` และ `cached`

ค่า availability นี้เป็น **ความครบถ้วนของข้อมูลจาก archive ที่เราตรวจสอบ** ไม่ใช่ตัวชี้วัดคุณภาพเครื่อง radiosonde โดยตรง

In [ ]:
# ================================================================
# CELL 8 — Availability summary and report
# ================================================================
success_status = {"downloaded", "cached"}
summary_rows = []

for hour in sorted(manifest["hour_utc"].dropna().unique()):
    m = manifest[manifest["hour_utc"] == hour].copy()
    n_requested = len(m)
    n_success = int(m["status"].isin(success_status).sum())
    n_no_data = int((m["status"] == "no_data").sum())
    n_error = int((m["status"] == "error").sum())
    availability_pct = 100.0 * n_success / n_requested if n_requested else np.nan

    summary_rows.append({
        "hour_utc": int(hour),
        "hour_ict": int((int(hour) + 7) % 24),
        "requested": n_requested,
        "success": n_success,
        "no_data": n_no_data,
        "error": n_error,
        "availability_pct": availability_pct,
    })

availability_summary = pd.DataFrame(summary_rows)
print("=== AVAILABILITY SUMMARY ===")
display(availability_summary)

print("\n=== STATUS COUNTS ===")
display(pd.crosstab(manifest["hour_utc"], manifest["status"], margins=True))

calendar_table = manifest.copy()
calendar_table["date"] = pd.to_datetime(calendar_table["date_utc"]).dt.strftime("%Y-%m-%d")
calendar_table["available"] = calendar_table["status"].isin(success_status).map({True: "OK", False: "—"})
calendar_pivot = calendar_table.pivot(index="date", columns="hour_utc", values="available")
calendar_pivot.columns = [f"{int(c):02d}Z" for c in calendar_pivot.columns]

print("\n=== DAILY AVAILABILITY ===")
display(calendar_pivot)

REPORT_FILE = META_DIR / f"download_report_{STATION_ID}_20240301_20240430.txt"
report_lines = [
    "Chiang Mai radiosonde download report",
    f"WMO station ID: {STATION_ID}",
    f"Expected station name: {STATION_NAME_EXPECTED}",
    f"Study period: {START_DATE} to {END_DATE}",
    f"UTC hours checked: {HOURS_UTC}",
    f"Source endpoint: {WYOMING_ENDPOINT}",
    "",
    "Availability summary:",
    availability_summary.to_string(index=False),
    "",
    "Status counts:",
    pd.crosstab(manifest["hour_utc"], manifest["status"], margins=True).to_string(),
]
REPORT_FILE.write_text("\n".join(report_lines), encoding="utf-8")
print("\nSaved report:", REPORT_FILE)

=== AVAILABILITY SUMMARY ===


,hour_utc,hour_ict,requested,success,no_data,error,availability_pct
0,0,7,61,61,0,0,100.0
1,6,13,61,0,0,61,0.0
2,12,19,61,0,0,61,0.0



=== STATUS COUNTS ===


status,cached,downloaded,error,All
hour_utc,,,,
0,1,60,0,61
6,0,0,61,61
12,0,0,61,61
All,1,60,122,183



=== DAILY AVAILABILITY ===


,00Z,06Z,12Z
date,,,
2024-03-01,OK,—,—
2024-03-02,OK,—,—
2024-03-03,OK,—,—
2024-03-04,OK,—,—
2024-03-05,OK,—,—
...,...,...,...
2024-04-26,OK,—,—
2024-04-27,OK,—,—
2024-04-28,OK,—,—



Saved report: /content/drive/MyDrive/Teaching_AirQuality/ChiangMai_Sounding_48327_MarApr2024/03_metadata/download_report_48327_20240301_20240430.txt


## 0.8 Combined long-format dataset

หนึ่ง sounding มีหลายระดับในแนวดิ่ง ดังนั้นเมื่อนำหลายวันมารวมกันจะใช้ **long format** เช่น

| launch_datetime_utc | pressure_hPa | height_m | temperature_C | ... |
|---|---:|---:|---:|---|
| 2024-03-01 00:00 | ... | ... | ... | ... |
| 2024-03-01 00:00 | ... | ... | ... | ... |
| 2024-03-02 00:00 | ... | ... | ... | ... |

ไฟล์รวมจะบีบอัดเป็น `.csv.gz` เพื่อให้ GitHub มีไฟล์น้อยและขนาดเล็ก โดย `pandas.read_csv()` อ่านไฟล์ gzip ได้โดยตรง

In [ ]:
# ================================================================
# CELL 9 — Build combined long-format dataset
# ================================================================
successful_manifest = manifest[manifest["status"].isin(success_status)].copy()
profile_frames = []

for _, row in successful_manifest.iterrows():
    csv_path = Path(row["profile_csv_file"])
    if not csv_path.exists():
        continue

    df = pd.read_csv(csv_path)
    df["source_file"] = csv_path.name
    df["source_archive"] = "University of Wyoming Atmospheric Science Radiosonde Archive"
    profile_frames.append(df)

if not profile_frames:
    raise RuntimeError("No successful sounding profiles were found. Inspect the manifest and test-launch output.")

combined = pd.concat(profile_frames, ignore_index=True)

preferred_order = [
    "station_id", "launch_datetime_utc", "launch_datetime_ict", "hour_utc", "hour_ict",
    "pressure_hPa", "height_m", "temperature_C", "dewpoint_C",
    "relative_humidity_pct", "mixing_ratio_gkg", "wind_direction_deg",
    "wind_speed_source", "wind_speed_source_unit", "wind_speed_ms",
    "theta_K", "theta_e_K", "theta_v_K", "source_file", "source_archive",
]
combined = combined[[c for c in preferred_order if c in combined.columns]]
combined = combined.sort_values(["launch_datetime_utc", "pressure_hPa"], ascending=[True, False]).reset_index(drop=True)

COMBINED_FILE = PROCESSED_DIR / f"chiangmai_{STATION_ID}_sounding_20240301_20240430.csv.gz"
combined.to_csv(COMBINED_FILE, index=False, compression="gzip")

print("Combined dataset saved:", COMBINED_FILE)
print("Shape:", combined.shape)
print("Unique successful launches:", combined["launch_datetime_utc"].nunique())
display(combined.head(12))

Combined dataset saved: /content/drive/MyDrive/Teaching_AirQuality/ChiangMai_Sounding_48327_MarApr2024/02_processed/chiangmai_48327_sounding_20240301_20240430.csv.gz
Shape: (4690, 20)
Unique successful launches: 61


,station_id,launch_datetime_utc,launch_datetime_ict,hour_utc,hour_ict,pressure_hPa,height_m,temperature_C,dewpoint_C,relative_humidity_pct,mixing_ratio_gkg,wind_direction_deg,wind_speed_source,wind_speed_source_unit,wind_speed_ms,theta_K,theta_e_K,theta_v_K,source_file,source_archive
0,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,0,7,1000.0,314.0,20.8,19.8,94.0,14.68,0.0,0.0,m/s,0.0,293.9,336.0,296.5,sounding_48327_2024-03-01_00Z.csv,University of Wyoming Atmospheric Science Radi...
1,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,0,7,968.0,597.0,22.4,14.4,61.0,10.71,NaN,NaN,m/s,NaN,298.3,329.7,300.2,sounding_48327_2024-03-01_00Z.csv,University of Wyoming Atmospheric Science Radi...
2,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,0,7,962.0,651.0,23.0,15.0,61.0,11.21,NaN,NaN,m/s,NaN,299.4,332.5,301.5,sounding_48327_2024-03-01_00Z.csv,University of Wyoming Atmospheric Science Radi...
3,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,0,7,957.0,696.0,23.6,14.6,57.0,10.98,NaN,NaN,m/s,NaN,300.5,333.0,302.5,sounding_48327_2024-03-01_00Z.csv,University of Wyoming Atmospheric Science Radi...
4,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,0,7,951.0,751.0,24.0,14.0,54.0,10.62,NaN,NaN,m/s,NaN,301.4,333.0,303.4,sounding_48327_2024-03-01_00Z.csv,University of Wyoming Atmospheric Science Radi...
5,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,0,7,946.0,798.0,24.4,13.4,50.0,10.26,NaN,NaN,m/s,NaN,302.3,333.0,304.2,sounding_48327_2024-03-01_00Z.csv,University of Wyoming Atmospheric Science Radi...
6,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,0,7,940.0,853.0,24.6,12.6,47.0,9.79,NaN,NaN,m/s,NaN,303.1,332.5,304.8,sounding_48327_2024-03-01_00Z.csv,University of Wyoming Atmospheric Science Radi...
7,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,0,7,935.0,900.0,25.0,11.0,41.0,8.85,NaN,NaN,m/s,NaN,303.9,330.7,305.6,sounding_48327_2024-03-01_00Z.csv,University of Wyoming Atmospheric Science Radi...
8,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,0,7,930.0,947.0,25.0,10.0,39.0,8.31,NaN,NaN,m/s,NaN,304.4,329.6,305.9,sounding_48327_2024-03-01_00Z.csv,University of Wyoming Atmospheric Science Radi...
9,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,0,7,925.0,994.0,25.2,10.2,39.0,8.47,NaN,NaN,m/s,NaN,305.1,330.8,306.6,sounding_48327_2024-03-01_00Z.csv,University of Wyoming Atmospheric Science Radi...


## 0.9 Station metadata

Metadata ควรแยกจาก code เพื่อให้ผู้ใช้ dataset ตรวจสอบ station ID, ช่วงเวลา, timezone, source และพิกัดที่ Wyoming report ได้

ถ้าบาง metadata ไม่ปรากฏใน response จะเก็บเป็น `NaN` แทนการเดาค่า

In [ ]:
# ================================================================
# CELL 10 — Build station metadata
# ================================================================
station_meta_df = pd.DataFrame(station_meta_records)


def first_valid(series, default=np.nan):
    if series is None:
        return default
    s = pd.Series(series).dropna()
    if len(s) == 0:
        return default
    mask = s.astype(str).str.strip() != ""
    s = s[mask]
    return s.iloc[0] if len(s) else default


station_metadata = {
    "station_id": STATION_ID,
    "station_name_expected": STATION_NAME_EXPECTED,
    "station_name_source": first_valid(station_meta_df.get("station_name_source")) if len(station_meta_df) else np.nan,
    "latitude": first_valid(station_meta_df.get("latitude")) if len(station_meta_df) else np.nan,
    "longitude": first_valid(station_meta_df.get("longitude")) if len(station_meta_df) else np.nan,
    "elevation_m": first_valid(station_meta_df.get("elevation_m")) if len(station_meta_df) else np.nan,
    "study_start": START_DATE,
    "study_end": END_DATE,
    "hours_utc_checked": ",".join(f"{int(h):02d}" for h in HOURS_UTC),
    "timezone_local": "Asia/Bangkok (UTC+7)",
    "source_name": "University of Wyoming Atmospheric Science Radiosonde Archive",
    "source_endpoint": WYOMING_ENDPOINT,
    "prepared_by_notebook": "00_download_ChiangMai_48327_Sounding_MarApr2024.ipynb",
}

station_metadata_df = pd.DataFrame([station_metadata])
STATION_META_FILE = META_DIR / f"station_metadata_{STATION_ID}.csv"
station_metadata_df.to_csv(STATION_META_FILE, index=False)

print("Station metadata saved:", STATION_META_FILE)
display(station_metadata_df)

Station metadata saved: /content/drive/MyDrive/Teaching_AirQuality/ChiangMai_Sounding_48327_MarApr2024/03_metadata/station_metadata_48327.csv


,station_id,station_name_expected,station_name_source,latitude,longitude,elevation_m,study_start,study_end,hours_utc_checked,timezone_local,source_name,source_endpoint,prepared_by_notebook
0,48327,Chiang Mai,"CHIANG MAI, THAILAND",18.78,98.98,NaN,2024-03-01,2024-04-30,"00,06,12",Asia/Bangkok (UTC+7),University of Wyoming Atmospheric Science Radi...,https://weather.uwyo.edu/wsgi/sounding,00_download_ChiangMai_48327_Sounding_MarApr202...


## 0.10 เตรียมไฟล์สำหรับ GitHub

แนะนำให้นิสิตใน Notebook 01–04 อ่านไฟล์ combined `.csv.gz` เพียงไฟล์เดียวจาก GitHub raw URL

ไฟล์ที่เตรียมใน `05_github_ready/` คือ

1. combined `.csv.gz` — dataset หลักสำหรับนิสิต
2. launch manifest — ตรวจ data availability
3. station metadata
4. download report
5. ZIP raw + individual profiles — สำหรับ provenance/reproducibility
6. `README_DATASET.md`

In [ ]:
# ================================================================
# CELL 11 — Create ZIP archive and GitHub-ready folder
# ================================================================
ARCHIVE_FILE = ARCHIVE_DIR / f"chiangmai_{STATION_ID}_sounding_20240301_20240430_raw_profiles.zip"
if ARCHIVE_FILE.exists():
    ARCHIVE_FILE.unlink()

with zipfile.ZipFile(ARCHIVE_FILE, mode="w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in sorted(RAW_DIR.glob("*.txt")):
        zf.write(file, arcname=f"00_raw_text/{file.name}")
    for file in sorted(PROFILE_DIR.glob("*.csv")):
        zf.write(file, arcname=f"01_profiles_csv/{file.name}")

print("Archive saved:", ARCHIVE_FILE)

for source in [COMBINED_FILE, MANIFEST_FILE, STATION_META_FILE, REPORT_FILE, ARCHIVE_FILE]:
    shutil.copy2(source, GITHUB_DIR / Path(source).name)

README_FILE = GITHUB_DIR / "README_DATASET.md"
availability_text = availability_summary.to_string(index=False)

readme_lines = [
    "# Chiang Mai Radiosonde Teaching Dataset",
    "",
    "## Dataset",
    "",
    f"- Station: Chiang Mai",
    f"- WMO ID: {STATION_ID}",
    f"- Period: {START_DATE} to {END_DATE}",
    f"- UTC hours checked: {HOURS_UTC}",
    "- Local timezone: ICT (UTC+7)",
    "- Source: University of Wyoming Atmospheric Science Radiosonde Archive",
    "",
    "## Availability",
    "",
    "```text",
    availability_text,
    "```",
    "",
    "## Recommended file for student notebooks",
    "",
    f"`{COMBINED_FILE.name}`",
    "",
    "Pandas can read the compressed CSV directly:",
    "",
    "```python",
    "import pandas as pd",
    "df = pd.read_csv(\"RAW_GITHUB_URL_HERE\", compression=\"gzip\")",
    "```",
    "",
    "## Provenance files",
    "",
    f"- `{MANIFEST_FILE.name}`",
    f"- `{STATION_META_FILE.name}`",
    f"- `{REPORT_FILE.name}`",
    f"- `{ARCHIVE_FILE.name}`",
    "",
    "Data source: University of Wyoming Atmospheric Science Radiosonde Archive.",
    "Retain source attribution and review current source data-use conditions before public redistribution.",
]

README_FILE.write_text("\n".join(readme_lines), encoding="utf-8")
print("Dataset README saved:", README_FILE)

Archive saved: /content/drive/MyDrive/Teaching_AirQuality/ChiangMai_Sounding_48327_MarApr2024/04_archive/chiangmai_48327_sounding_20240301_20240430_raw_profiles.zip
Dataset README saved: /content/drive/MyDrive/Teaching_AirQuality/ChiangMai_Sounding_48327_MarApr2024/05_github_ready/README_DATASET.md


## 0.11 ตรวจขนาดไฟล์และ hash ก่อน GitHub

เกณฑ์เชิงปฏิบัติใน cell ต่อไปช่วยเตือนเรื่องขนาดไฟล์

- `< 25 MB` — เหมาะกับ browser upload โดยทั่วไป
- `25–50 MB` — ควรใช้ Git command line
- `50–100 MB` — ค่อนข้างใหญ่ ควรพิจารณาบีบอัด/จัดโครงสร้างใหม่
- `> 100 MB` — ไม่เหมาะกับ regular Git file; พิจารณา Git LFS หรือ data hosting อื่น

สำหรับ sounding สถานีเดียวสองเดือน combined `.csv.gz` ควรเล็กกว่าขีดจำกัดเหล่านี้มาก

In [ ]:
# ================================================================
# CELL 12 — Final validation, file sizes and SHA-256
# ================================================================
import hashlib


def sha256_file(path: Path, chunk_size=1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def github_size_note(size_mb: float) -> str:
    if size_mb < 25:
        return "OK for normal browser upload"
    if size_mb < 50:
        return "Prefer Git command line"
    if size_mb <= 100:
        return "Large file: reconsider compression/repository design"
    return "Too large for regular Git file; consider Git LFS/other hosting"


inventory = []
for file in sorted(GITHUB_DIR.iterdir()):
    if not file.is_file():
        continue
    size_mb = file.stat().st_size / (1024 ** 2)
    inventory.append({
        "file": file.name,
        "size_MB": round(size_mb, 3),
        "github_note": github_size_note(size_mb),
        "sha256": sha256_file(file),
    })

file_inventory_df = pd.DataFrame(inventory)
print("=== GITHUB-READY FILES ===")
display(file_inventory_df)

print("\nFolder to upload:")
print(GITHUB_DIR)

print("\n=== DOWNLOAD SUMMARY TO REPORT BACK ===")
display(availability_summary)

=== GITHUB-READY FILES ===


,file,size_MB,github_note,sha256
0,README_DATASET.md,0.001,OK for normal browser upload,a3fbb19bcbcb04fc24ecd7d16a379a890e5e2f86b7f258...
1,chiangmai_48327_sounding_20240301_20240430.csv.gz,0.114,OK for normal browser upload,09a6adc855c64af30ad9ae7704ddf2064bb3f8024ab8e7...
2,chiangmai_48327_sounding_20240301_20240430_raw...,0.398,OK for normal browser upload,94bf8173fa5cd8179f6a7affb6d0f187eeccfc461ef51a...
3,download_report_48327_20240301_20240430.txt,0.001,OK for normal browser upload,923caa33bc7c9a1bacdb7ed10f760cc50d3a52df62b6fb...
4,launch_manifest_48327_20240301_20240430.csv,0.070,OK for normal browser upload,61d1357b249f7fc76c8745b8434a1ea424fb6e219a1294...
5,station_metadata_48327.csv,0.000,OK for normal browser upload,75a958df10a95209e8dafbbf29de1422e4ec28024f7742...



Folder to upload:
/content/drive/MyDrive/Teaching_AirQuality/ChiangMai_Sounding_48327_MarApr2024/05_github_ready

=== DOWNLOAD SUMMARY TO REPORT BACK ===


,hour_utc,hour_ict,requested,success,no_data,error,availability_pct
0,0,7,61,61,0,0,100.0
1,6,13,61,0,0,61,0.0
2,12,19,61,0,0,61,0.0


# หลังรัน Notebook 00 ให้ตรวจ 4 ประเด็น

1. **00 UTC success กี่วันจาก 61 วัน**
2. **06 UTC มี sounding จริงหรือไม่** — ถ้ามี เท่ากับประมาณ 13:00 ICT และมีคุณค่ามากต่อการเปรียบเทียบ daytime boundary layer
3. **12 UTC มี sounding จริงหรือไม่**
4. combined `.csv.gz` และ ZIP มีขนาดเท่าใด

ผลที่ควรนำกลับมาใช้วาง Notebook 01 คือสองตารางท้าย notebook:

- **AVAILABILITY SUMMARY**
- **GITHUB-READY FILES**

เมื่ออัปโหลดไฟล์ใน `05_github_ready/` ขึ้น GitHub แล้ว Notebook 01–04 จะอ่าน combined dataset จาก **GitHub raw URL** โดยตรง เพื่อให้นิสิตทุกคนใช้ข้อมูลชุดเดียวกัน